# Sahayak — Fine-tune Gemma 4 **E2B** (QLoRA) on Kaggle **GPU T4 ×2**

End-to-end: install → GPU guard → HF auth → dataset validate (hard gate) → **E2B QLoRA** train →
merged-weights export → upload to a private HF repo. Companion to `docs/SAHAYAK_DATASET_SPEC.md`
and `finetune/`. **Pure `transformers` + `peft` + `bitsandbytes` — no Unsloth.**

### Before you run
1. **Settings → Accelerator = `GPU T4 x2`** (never P100 — compute capability 6.0 lacks the bitsandbytes 4-bit kernels; T4 = 7.5). **Internet = On**.
2. **Add-ons → Secrets → add `HF_TOKEN`** = a Hugging Face token whose account has **accepted the Gemma license** at [huggingface.co/google/gemma-4-E2B-it](https://huggingface.co/google/gemma-4-E2B-it). *Never paste the token in a cell.*
3. **Add Input → your dataset** containing `train.jsonl`, `val.jsonl`, and (optional) `eval_holdout.jsonl`. Cell 5 auto-finds them under `/kaggle/input`.

The script is single-GPU; the **second T4 idles** (harmless). Precision on a T4: Gemma 4 has
fp16-unsafe ops and the T4 has no native bf16, so the trainer runs **4-bit NF4 weights + float32
compute** — slower than bf16 but numerically safe. E2B fits this path on one 16 GB T4; use E4B
only on a native-bf16 GPU (Colab L4/A100), where the trainer switches to bf16 automatically.


## Cell 1 — install the fine-tune stack

Kaggle's image already ships torch + CUDA; we add the pinned Apache-2.0 stack from the repo's
`finetune/requirements.txt`: `transformers` (≥ 5.6 for native Gemma 4), `datasets`, `peft`,
`accelerate`, `bitsandbytes`. No Unsloth, no TRL.

> If pip prints a torch/transformers conflict, do **Run → Restart & clear cell outputs**, then re-run from Cell 1.


In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

# Same pins as finetune/requirements.txt (repo isn't cloned yet at this point).
pip('transformers>=5.6,<6', 'datasets>=3.6,<6', 'peft>=0.16,<1',
    'accelerate>=1.6,<2', 'bitsandbytes>=0.46,<1')
# For the artifact upload step.
pip('-U', 'huggingface_hub[hf_transfer]')
print('\ninstall complete')


## Cell 2 — GPU guard (reject P100, confirm T4)

Fails loudly *before* a long run if the accelerator is wrong. bitsandbytes 4-bit needs CUDA compute capability ≥ 7.0.


In [ ]:
import torch, subprocess

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

assert torch.cuda.is_available(), "No CUDA GPU visible. Settings -> Accelerator -> 'GPU T4 x2'."
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name}   compute capability: {major}.{minor}   VRAM: {vram:.1f} GB')
assert major >= 7, (
    f'Compute capability {major}.{minor} < 7.0 (e.g. Tesla P100 = 6.0). bitsandbytes 4-bit '
    "needs >= 7.0. Switch the accelerator to 'GPU T4 x2' (T4 = 7.5)."
)
print('GPU OK for QLoRA.')


## Cell 3 — Hugging Face auth (from Kaggle Secrets)

The token is read from the `HF_TOKEN` **secret** — it is never written into the notebook or any file.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
    print('HF_TOKEN loaded from Kaggle Secrets.')
except Exception as e:
    raise SystemExit(
        'Could not load HF_TOKEN from Kaggle Secrets. Add-ons -> Secrets -> add HF_TOKEN '
        '(a token whose HF account has accepted the Gemma license at '
        'huggingface.co/google/gemma-4-E2B-it). Original error: ' + repr(e)
    )

## Cell 4 — get the training code

Clones the repo for `sahayak_finetune.py` + `validate_dataset.py` and `cd`s into `finetune/` (so the
validator can import the canonical `SYSTEM_PROMPT` from the trainer). If your repo is private, either
make it public for the clone or upload the `finetune/` folder as a Kaggle dataset and point `FT` at it.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/SivaNithishKumar/sankat-mochan.git'
WORK = '/kaggle/working'
repo_dir = os.path.join(WORK, 'sankat-mochan')
if not os.path.isdir(repo_dir):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, repo_dir], check=True)
else:
    # Re-runs in the same session pick up pushed fixes instead of training stale code.
    subprocess.run(['git', '-C', repo_dir, 'pull', '--ff-only'], check=True)

FT = os.path.join(repo_dir, 'finetune')
assert os.path.exists(os.path.join(FT, 'sahayak_finetune.py')), 'trainer script missing after clone'
assert os.path.exists(os.path.join(FT, 'validate_dataset.py')), 'validator missing after clone'
os.chdir(FT)
print('finetune dir:', FT)


## Cell 5 — locate the dataset & validate (HARD GATE)

Auto-finds the JSONL under `/kaggle/input`. The mechanical validator (`SAHAYAK_DATASET_SPEC.md` §8.1) is a
**hard gate** — training never runs on a file that fails it. `--eval` uses the **validation split**
(`val.jsonl`), *not* the held-out `eval_holdout.jsonl` (that is for the final hand-review only).

In [ ]:
import glob, os, subprocess, sys

def find_one(names):
    for n in names:
        hits = sorted(glob.glob(f'/kaggle/input/**/{n}', recursive=True))
        if hits:
            return hits[0]
    return None

TRAIN = find_one(['train.jsonl', 'all.jsonl'])
VAL   = find_one(['val.jsonl'])
HOLD  = find_one(['eval_holdout.jsonl', 'holdout.jsonl'])
print('train  :', TRAIN)
print('val    :', VAL)
print('holdout:', HOLD)
assert TRAIN and VAL, 'Attach a Kaggle dataset with train.jsonl and val.jsonl (Add Input).'

for f in [TRAIN, VAL]:
    rc = subprocess.run([sys.executable, 'validate_dataset.py', f]).returncode
    assert rc == 0, f'validation FAILED for {f} - fix the data before training.'
print('\ndataset validation passed.')

## Cell 6 — train E2B (QLoRA)

Best-accuracy settings from the project guide: **r=32 / α=32**, **lr 2e-4**, **3 epochs**, effective batch
**8** (1 × grad-accum 8), **seq-len 1024**. Batch size 1 is deliberate on a T4: float32 compute means the
cross-entropy over Gemma's ~262k vocab materializes ~1 GB of logits **per sequence** — batch 2 OOMs at
step 0 (tested). Assistant-only loss masking is built into the trainer (character-offset masking against
the model's own chat template — the same template the on-device runtime must use). Watch for a falling
**eval** loss each epoch.

Still OOM? Add `'--device-map', 'balanced'` to the command — it shards the model's layers across both
T4s (naive model parallelism: halves per-GPU memory, but GPUs take turns so it's ~1.5–2× slower).
Use it as a memory escape hatch, not a speed-up.

_This runs the full training; expect ~1.5–2 h including the merged export._


In [ ]:
import subprocess, sys

OUT = '/kaggle/working/sahayak-e2b'
cmd = [
    sys.executable, 'sahayak_finetune.py',
    '--train', TRAIN,
    '--eval',  VAL,
    '--model', 'google/gemma-4-E2B-it',
    '--out',   OUT,
    '--epochs', '3',
    '--lr', '2e-4',
    '--batch-size', '1',
    '--grad-accum', '8',
    '--lora-r', '32',
    '--lora-alpha', '32',
    '--max-seq-len', '1024',
    '--export-merged',
]
print(' '.join(cmd), '\n')
subprocess.run(cmd, check=True)


> **On exports:** LoRA adapters (`OUT/`) are saved *before* the merge, so they are safe even if the
> merge step fails. `OUT/merged/` holds the full merged weights — the **Qualcomm AI Hub** input. For a
> GGUF (llama.cpp CPU/GPU path), run llama.cpp's `convert_hf_to_gguf.py` (MIT-licensed) against
> `OUT/merged/` afterwards — GGUF conversion is deliberately out of this notebook.


## Cell 7 — quick qualitative check on the holdout _(optional)_

Loads the fine-tuned adapters and generates a few held-out answers with Gemma 4's recommended sampling
(`temp 1.0, top_p 0.95, top_k 64`). Eyeball packet format, refusal calibration, length, and language.

In [ ]:
import json, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sahayak_finetune import SYSTEM_PROMPT

BASE = 'google/gemma-4-E2B-it'
tok = AutoTokenizer.from_pretrained(OUT)
base = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float32,
    ),
    device_map={'': 0},
    attn_implementation='eager',
)
model = PeftModel.from_pretrained(base, OUT)
model.eval()

prompts = []
if HOLD:
    with open(HOLD, encoding='utf-8') as f:
        for line in f:
            o = json.loads(line)
            u = next((m['content'] for m in o['messages'] if m['role'] == 'user'), None)
            if u:
                prompts.append(u)
            if len(prompts) >= 3:
                break

for u in prompts:
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': u}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(input_ids=ids, max_new_tokens=256, do_sample=True,
                             temperature=1.0, top_p=0.95, top_k=64)
    print('USER    :', u)
    print('SAHAYAK :', tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
    print('-' * 80)


## Cell 8 — ship artifacts to a private HF repo

Uploads `adapters + merged-16bit + gguf` to `&lt;your-hf-user&gt;/sahayak-e2b` (private). The repo owner is
resolved from your token, so no username is hard-coded.

In [ ]:
import os
from huggingface_hub import HfApi

api = HfApi(token=os.environ['HF_TOKEN'])
user = api.whoami()['name']
repo = f'{user}/sahayak-e2b'
api.create_repo(repo, private=True, exist_ok=True)
api.upload_folder(folder_path=OUT, repo_id=repo)
print('uploaded ->', repo)
print('pull locally:  huggingface-cli download', repo, '--local-dir ./sahayak-e2b')

## Artifacts & next step (NPU)

You get two artifacts — keep both:
- `merged/` (safetensors) → the **Qualcomm AI Hub** input for NPU compilation, and the input for a
  later GGUF conversion (llama.cpp `convert_hf_to_gguf.py`) for the app's llama.cpp CPU/GPU path.
- adapter files at the root (~100–200 MB) → re-merge later without retraining.

**Inference settings to bake into the app (must match at demo):** `temperature=1.0, top_p=0.95, top_k=64`,
the **model's own chat template** (saved with the tokenizer in `OUT/`), and the **byte-identical**
system prompt from the spec.

**NPU handoff (not on Kaggle):** AI Hub's LLM flow consumes the HF checkpoint (`merged/`), never GGUF —
quantize (AIMET, Linux/WSL + big-VRAM GPU) then export to QNN/Genie for `Snapdragon X Elite CRD`. See the
project's Kaggle guide §5 and `PLAN.md`.
